# 08 - Leave-One-Corpus-Out (LOCO) 跨語料庫實驗 (Colab T4 GPU)

**實驗設計**：4 輪 LOCO — 每次留一個 dataset 作 test，其餘 3 個作 train。

| 輪次 | Train (80%) + Val (20%) | Test (完全不碰) |
|------|------------------------|----------------|
| 1 | CREMA-D, TESS, SAVEE | RAVDESS |
| 2 | RAVDESS, TESS, SAVEE | CREMA-D |
| 3 | RAVDESS, CREMA-D, SAVEE | TESS |
| 4 | RAVDESS, CREMA-D, TESS | SAVEE |

**Early Stopping 機制**：
- 從 3 個 train corpus 中，用 StratifiedGroupKFold (group=speaker_id) 再分出 train/val
- **Val set** 用於 early stopping 和 checkpoint 選擇
- **Test corpus** 在訓練過程中完全不碰，只做最終評估

**支援分段執行**：每個模型的結果獨立存檔，斷線可跳過已完成的 run。

**前置需求**：先執行 05/06/07 notebook 取得 in-corpus 結果 JSON。

In [ ]:
# === 安裝相依套件 ===
!pip install -q transformers datasets accelerate peft librosa soundfile plotly kaleido
!pip install -q "kaleido==0.2.1"

In [ ]:
# === 路徑設定（支援 Colab 與本地執行）===
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/SER-Project')
except (ImportError, ModuleNotFoundError):
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

sys.path.append(str(PROJECT_ROOT / 'src'))

RESULTS_DIR = PROJECT_ROOT / 'results'
LOCO_DIR = RESULTS_DIR / 'cross_corpus'
FIGURES_DIR = RESULTS_DIR / 'figures'
CKPT_DIR = PROJECT_ROOT / 'models' / 'checkpoints' / 'loco'
LOCO_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
import torch
print(f'Device: {"GPU (" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "CPU"}')

In [ ]:
# === Imports ===
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor
from tqdm.auto import tqdm

from models import SERConvNet, SERBiLSTM, EarlyStopping

# === 常數 ===
RANDOM_STATE = 42
TARGET_SR = 16000
MAX_LENGTH_SAMPLES = 48000
MAX_GRAD_NORM = 1.0
VAL_SPLITS = 5  # 從 train pool 取 1/5 作 validation
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASETS = ['RAVDESS', 'CREMA-D', 'TESS', 'SAVEE']

# === 可重現性 ===
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'Device: {DEVICE}')

In [ ]:
# === 載入 metadata ===
df = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
le = LabelEncoder()
le.fit(df['emotion'])
classes = le.classes_.tolist()

print(f'Total samples: {len(df):,}')
print(f'Classes: {classes}')
print(f'\nPer-dataset counts:')
print(df['dataset'].value_counts().to_string())

In [ ]:
# === 複製音訊到 Colab 本地暫存（wav2vec 用，避免 Drive I/O 瓶頸） ===
LOCAL_AUDIO = Path('/content/temp_audio')

if LOCAL_AUDIO.exists() and len(list(LOCAL_AUDIO.glob('*.wav'))) > 11000:
    print(f'Local cache exists: {len(list(LOCAL_AUDIO.glob("*.wav"))):,} files')
else:
    AUDIO_DIR = PROJECT_ROOT / 'data' / 'processed' / 'audio_16k'
    print('Copying audio files to local storage (one-time, ~1-2 min)...')
    LOCAL_AUDIO.mkdir(exist_ok=True)
    !cp "{AUDIO_DIR}"/*.wav /content/temp_audio/
    n_files = len(list(LOCAL_AUDIO.glob('*.wav')))
    print(f'Done: {n_files:,} files copied to {LOCAL_AUDIO}')

In [ ]:
# === Dataset 類別 ===

class MelSpecDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths, self.labels = paths, labels
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        mel = np.load(self.paths[idx])[np.newaxis, :, :]  # (1, 128, 94)
        return torch.tensor(mel, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)


class MFCCDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths, self.labels = paths, labels
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        mfcc = np.load(self.paths[idx])  # (94, 39)
        return torch.tensor(mfcc, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)


class AudioDataset(Dataset):
    def __init__(self, paths, labels, feature_extractor, max_length=MAX_LENGTH_SAMPLES):
        self.paths, self.labels = paths, labels
        self.fe = feature_extractor
        self.max_length = max_length
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        audio, _ = librosa.load(self.paths[idx], sr=TARGET_SR)
        if len(audio) > self.max_length:
            start = (len(audio) - self.max_length) // 2
            audio = audio[start:start + self.max_length]
        elif len(audio) < self.max_length:
            pad_total = self.max_length - len(audio)
            audio = np.pad(audio, (pad_total // 2, pad_total - pad_total // 2))
        inputs = self.fe(audio, sampling_rate=TARGET_SR, return_tensors='pt', padding=False)
        return inputs['input_values'].squeeze(0), torch.tensor(self.labels[idx], dtype=torch.long)

print('Dataset classes defined.')

In [ ]:
# === 通用訓練 / 評估 / 結果管理函式 ===

def train_one_epoch_standard(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(X)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total


def train_one_epoch_wav2vec(model, loader, criterion, optimizer, scaler, device, grad_accum=8):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()
    for step, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)
        with torch.cuda.amp.autocast():
            logits = model(X).logits
            loss = criterion(logits, y)
            scaled_loss = loss / grad_accum
        scaler.scale(scaled_loss).backward()
        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_model(model, loader, criterion, device, is_wav2vec=False):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        with torch.cuda.amp.autocast():
            logits = model(X).logits if is_wav2vec else model(X)
            loss = criterion(logits, y)
        total_loss += loss.item() * len(y)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    preds, labels = np.array(all_preds), np.array(all_labels)
    return {
        'loss': total_loss / len(labels),
        'accuracy': accuracy_score(labels, preds),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'preds': preds, 'labels': labels,
    }


def load_existing_results(path: Path) -> dict:
    """載入已存在的結果 JSON，若不存在則回傳空 dict。"""
    if path.exists():
        with open(path, encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_results_incremental(path: Path, key: str, result: dict):
    """增量儲存結果：讀取 → 更新 → 寫回。"""
    existing = load_existing_results(path)
    existing[key] = result
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(existing, f, indent=2, ensure_ascii=False, default=str)

print('Training/evaluation functions defined.')

In [ ]:
# === LOCO 單輪訓練函式 ===
# 關鍵設計：從 3 個 train corpus 分出獨立 val set，test corpus 完全不碰

def _build_paths_and_ds(model_type, sub_df, le, fe=None):
    """根據模型類型建立路徑和 Dataset。"""
    y = le.transform(sub_df['emotion'].values)
    if model_type == 'cnn':
        paths = [str(PROJECT_ROOT / p) for p in sub_df['melspec_path']]
        ds = MelSpecDataset(paths, y)
    elif model_type == 'lstm':
        paths = [str(PROJECT_ROOT / p) for p in sub_df['mfcc_path']]
        ds = MFCCDataset(paths, y)
    elif model_type == 'wav2vec':
        paths = [str(LOCAL_AUDIO / Path(p).name) for p in sub_df['processed_path']]
        ds = AudioDataset(paths, y, fe)
    return ds, y


def run_loco_one(model_type: str, pool_df: pd.DataFrame, test_df: pd.DataFrame,
                 le: LabelEncoder, test_corpus: str) -> dict:
    """對指定模型跑一輪 LOCO。

    pool_df: 3 個 train corpus 合併，會再從中分出 train/val。
    test_df: 留出的 test corpus，訓練過程中完全不碰。
    """
    num_classes = len(le.classes_)

    # === Step 1: 從 pool 中分出 train (80%) 和 val (20%) ===
    # 使用 StratifiedGroupKFold，group=speaker_id，確保同一 speaker 不跨 train/val
    y_pool = le.transform(pool_df['emotion'].values)
    groups_pool = pool_df['speaker_id'].values

    sgkf_val = StratifiedGroupKFold(n_splits=VAL_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    train_idx, val_idx = next(sgkf_val.split(np.zeros(len(y_pool)), y_pool, groups_pool))

    train_df = pool_df.iloc[train_idx].reset_index(drop=True)
    val_df = pool_df.iloc[val_idx].reset_index(drop=True)

    train_speakers = set(pool_df.iloc[train_idx]['speaker_id'])
    val_speakers = set(pool_df.iloc[val_idx]['speaker_id'])
    assert len(train_speakers & val_speakers) == 0, 'Speaker leakage between train/val!'

    print(f'    Pool split: train={len(train_df):,} ({len(train_speakers)} spk), '
          f'val={len(val_df):,} ({len(val_speakers)} spk), '
          f'test={len(test_df):,} (held-out corpus)')

    # === Step 2: Class weight（只從 actual train 計算）===
    y_train = le.transform(train_df['emotion'].values)
    cw = compute_class_weight('balanced', classes=np.arange(num_classes), y=y_train)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(DEVICE))

    # === Step 3: 建立 3 組 Dataset + DataLoader ===
    fe = None
    if model_type == 'wav2vec':
        fe = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base')

    train_ds, _ = _build_paths_and_ds(model_type, train_df, le, fe)
    val_ds, _ = _build_paths_and_ds(model_type, val_df, le, fe)
    test_ds, _ = _build_paths_and_ds(model_type, test_df, le, fe)

    if model_type == 'cnn':
        batch_size, epochs, patience = 32, 50, 7
        model = SERConvNet(num_classes).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        is_wav2vec = False
    elif model_type == 'lstm':
        batch_size, epochs, patience = 32, 50, 7
        model = SERBiLSTM(input_dim=39, hidden_dim=128, num_layers=2, num_classes=num_classes).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        is_wav2vec = False
    elif model_type == 'wav2vec':
        batch_size, epochs, patience = 4, 20, 5
        model = Wav2Vec2ForSequenceClassification.from_pretrained(
            'facebook/wav2vec2-base', num_labels=num_classes, classifier_proj_size=256)
        model.freeze_feature_encoder()
        for i in range(8):
            for param in model.wav2vec2.encoder.layers[i].parameters():
                param.requires_grad = False
        model = model.to(DEVICE)
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5, weight_decay=0.01)
        is_wav2vec = True
    else:
        raise ValueError(f'Unknown: {model_type}')

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    sched_patience = 2 if is_wav2vec else 3
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=sched_patience)
    scaler = torch.cuda.amp.GradScaler()
    early_stopping = EarlyStopping(patience=patience)
    best_val_loss = float('inf')
    ckpt_path = CKPT_DIR / f'{model_type}_test_{test_corpus}.pt'
    t_start = time.time()

    # === Step 4: Training loop — early stopping 用 val（非 test）===
    for epoch in range(epochs):
        if is_wav2vec:
            train_loss, _ = train_one_epoch_wav2vec(
                model, train_loader, criterion, optimizer, scaler, DEVICE, grad_accum=8)
        else:
            train_loss, _ = train_one_epoch_standard(
                model, train_loader, criterion, optimizer, scaler, DEVICE)

        # ★ 用 val_loader 做 early stopping，不碰 test_loader
        val_result = evaluate_model(model, val_loader, criterion, DEVICE, is_wav2vec=is_wav2vec)
        val_loss = val_result['loss']
        scheduler.step(val_loss)
        early_stopping(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), ckpt_path)

        if (epoch + 1) % 5 == 0 or early_stopping.should_stop:
            print(f'    Epoch {epoch+1}: val_loss={val_loss:.4f}, val_acc={val_result["accuracy"]:.4f}, '
                  f'val_F1(m)={val_result["f1_macro"]:.4f}')
        if early_stopping.should_stop:
            print(f'    Early stopping at epoch {epoch+1}')
            break

    # === Step 5: 載入最佳權重，在 test corpus 上做最終評估 ===
    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    final = evaluate_model(model, test_loader, criterion, DEVICE, is_wav2vec=is_wav2vec)
    elapsed = time.time() - t_start
    cm = confusion_matrix(final['labels'], final['preds']).tolist()
    report = classification_report(
        final['labels'], final['preds'], target_names=le.classes_.tolist(), output_dict=True)

    print(f'    [TEST] acc={final["accuracy"]:.4f}, F1(m)={final["f1_macro"]:.4f} (on held-out {test_corpus})')

    del model, optimizer, scaler, criterion
    torch.cuda.empty_cache()

    return {
        'accuracy': final['accuracy'], 'f1_weighted': final['f1_weighted'],
        'f1_macro': final['f1_macro'], 'confusion_matrix': cm,
        'classification_report': report, 'stopped_epoch': epoch + 1,
        'time_sec': round(elapsed, 1),
        'train_samples': len(train_df), 'val_samples': len(val_df),
        'test_samples': len(test_df),
    }

print('LOCO experiment function defined.')

---
## Section A: CNN LOCO (可獨立執行)
跑完後結果存到 `results/cross_corpus/loco_cnn.json`

In [ ]:
RESULT_PATH = LOCO_DIR / 'loco_cnn.json'
existing = load_existing_results(RESULT_PATH)

for test_corpus in DATASETS:
    key = f'test_{test_corpus}'
    if key in existing:
        print(f'[SKIP] CNN {key} already exists '
              f'(acc={existing[key]["accuracy"]:.4f})')
        continue

    print(f'\n--- CNN LOCO: test={test_corpus} ---')
    pool_df = df[df['dataset'] != test_corpus].reset_index(drop=True)
    test_df = df[df['dataset'] == test_corpus].reset_index(drop=True)
    print(f'  Pool (3 corpora): {len(pool_df):,} | Test (held-out): {len(test_df):,}')

    result = run_loco_one('cnn', pool_df, test_df, le, test_corpus)
    save_results_incremental(RESULT_PATH, key, result)
    print(f'  [SAVED] acc={result["accuracy"]:.4f}, F1(m)={result["f1_macro"]:.4f}')

print(f'\nCNN LOCO complete. File: {RESULT_PATH}')

---
## Section B: LSTM LOCO (可獨立執行)
跑完後結果存到 `results/cross_corpus/loco_lstm.json`

In [ ]:
RESULT_PATH = LOCO_DIR / 'loco_lstm.json'
existing = load_existing_results(RESULT_PATH)

for test_corpus in DATASETS:
    key = f'test_{test_corpus}'
    if key in existing:
        print(f'[SKIP] LSTM {key} already exists '
              f'(acc={existing[key]["accuracy"]:.4f})')
        continue

    print(f'\n--- LSTM LOCO: test={test_corpus} ---')
    pool_df = df[df['dataset'] != test_corpus].reset_index(drop=True)
    test_df = df[df['dataset'] == test_corpus].reset_index(drop=True)
    print(f'  Pool (3 corpora): {len(pool_df):,} | Test (held-out): {len(test_df):,}')

    result = run_loco_one('lstm', pool_df, test_df, le, test_corpus)
    save_results_incremental(RESULT_PATH, key, result)
    print(f'  [SAVED] acc={result["accuracy"]:.4f}, F1(m)={result["f1_macro"]:.4f}')

print(f'\nLSTM LOCO complete. File: {RESULT_PATH}')

---
## Section C: wav2vec LOCO (可獨立執行)
跑完後結果存到 `results/cross_corpus/loco_wav2vec.json`

> 注意：wav2vec 每輪約 15-30 分鐘，4 輪共需 1-2 小時。

In [ ]:
RESULT_PATH = LOCO_DIR / 'loco_wav2vec.json'
existing = load_existing_results(RESULT_PATH)

for test_corpus in DATASETS:
    key = f'test_{test_corpus}'
    if key in existing:
        print(f'[SKIP] wav2vec {key} already exists '
              f'(acc={existing[key]["accuracy"]:.4f})')
        continue

    print(f'\n--- wav2vec LOCO: test={test_corpus} ---')
    pool_df = df[df['dataset'] != test_corpus].reset_index(drop=True)
    test_df = df[df['dataset'] == test_corpus].reset_index(drop=True)
    print(f'  Pool (3 corpora): {len(pool_df):,} | Test (held-out): {len(test_df):,}')

    result = run_loco_one('wav2vec', pool_df, test_df, le, test_corpus)
    save_results_incremental(RESULT_PATH, key, result)
    print(f'  [SAVED] acc={result["accuracy"]:.4f}, F1(m)={result["f1_macro"]:.4f}')

print(f'\nwav2vec LOCO complete. File: {RESULT_PATH}')

---
## 結果匯總與視覺化
以下 cell 讀取三個獨立 JSON，合併產出總表與圖表。

In [ ]:
# === 載入三個模型的 LOCO 結果 ===
MODEL_TYPES = ['cnn', 'lstm', 'wav2vec']
all_loco = {}

for mt in MODEL_TYPES:
    path = LOCO_DIR / f'loco_{mt}.json'
    if path.exists():
        with open(path, encoding='utf-8') as f:
            all_loco[mt] = json.load(f)
        print(f'{mt}: {len(all_loco[mt])} corpus results loaded')
    else:
        print(f'[WARNING] {path.name} not found — skip {mt}')

# 總表
rows = []
for mt in MODEL_TYPES:
    if mt not in all_loco:
        continue
    for test_corpus in DATASETS:
        key = f'test_{test_corpus}'
        if key not in all_loco[mt]:
            continue
        r = all_loco[mt][key]
        rows.append({
            'Model': mt.upper(),
            'Test Corpus': test_corpus,
            'Accuracy': f"{r['accuracy']:.4f}",
            'F1 (weighted)': f"{r['f1_weighted']:.4f}",
            'F1 (macro)': f"{r['f1_macro']:.4f}",
            'Time (s)': r['time_sec'],
        })

loco_df = pd.DataFrame(rows)
loco_df.to_csv(LOCO_DIR / 'loco_summary.csv', index=False)
print('\nLOCO Summary:')
print(loco_df.to_string(index=False))

In [ ]:
# === In-corpus vs Cross-corpus 比較 ===
comparison_rows = []

for mt, json_name in [('cnn', 'cnn_results.json'),
                       ('lstm', 'lstm_results.json'),
                       ('wav2vec', 'wav2vec_results.json')]:
    # In-corpus
    json_path = RESULTS_DIR / json_name
    if json_path.exists():
        with open(json_path, encoding='utf-8') as f:
            ic = json.load(f)
        ic_acc = ic['summary']['accuracy_mean']
        ic_f1m = ic['summary']['f1_macro_mean']
    else:
        ic_acc, ic_f1m = None, None
        print(f'[WARNING] {json_name} not found')

    # Cross-corpus (LOCO mean)
    if mt in all_loco:
        accs = [all_loco[mt][f'test_{c}']['accuracy'] for c in DATASETS if f'test_{c}' in all_loco[mt]]
        f1ms = [all_loco[mt][f'test_{c}']['f1_macro'] for c in DATASETS if f'test_{c}' in all_loco[mt]]
        cc_acc, cc_f1m = np.mean(accs), np.mean(f1ms)
    else:
        cc_acc, cc_f1m = None, None

    comparison_rows.append({
        'Model': mt.upper(),
        'In-corpus Acc': f'{ic_acc:.4f}' if ic_acc else 'N/A',
        'Cross-corpus Acc': f'{cc_acc:.4f}' if cc_acc else 'N/A',
        'Acc Drop': f'{ic_acc - cc_acc:.4f}' if (ic_acc and cc_acc) else 'N/A',
        'In-corpus F1(m)': f'{ic_f1m:.4f}' if ic_f1m else 'N/A',
        'Cross-corpus F1(m)': f'{cc_f1m:.4f}' if cc_f1m else 'N/A',
        'F1(m) Drop': f'{ic_f1m - cc_f1m:.4f}' if (ic_f1m and cc_f1m) else 'N/A',
    })

comp_df = pd.DataFrame(comparison_rows)
comp_df.to_csv(LOCO_DIR / 'in_vs_cross_corpus.csv', index=False)
print('\nIn-corpus vs Cross-corpus:')
print(comp_df.to_string(index=False))

In [ ]:
# === 視覺化：Cross-corpus accuracy per model ===
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[mt.upper() for mt in MODEL_TYPES if mt in all_loco],
    horizontal_spacing=0.08,
)

for col_idx, mt in enumerate([m for m in MODEL_TYPES if m in all_loco], 1):
    accs = [all_loco[mt].get(f'test_{c}', {}).get('accuracy', 0) for c in DATASETS]
    fig.add_trace(
        go.Bar(x=DATASETS, y=accs, name=mt.upper(),
               text=[f'{a:.3f}' for a in accs], textposition='outside',
               showlegend=False),
        row=1, col=col_idx,
    )
    fig.update_yaxes(range=[0, 1], row=1, col=col_idx)

fig.update_layout(
    title='LOCO Cross-corpus Accuracy (test on each corpus)',
    height=450, width=1100, template='plotly_white',
)
fig.write_html(FIGURES_DIR / '10_loco_accuracy.html')
fig.write_image(FIGURES_DIR / '10_loco_accuracy.png', width=1100, height=450, scale=2)
fig.show()

In [ ]:
# === 視覺化：In-corpus vs Cross-corpus F1 比較 ===
ic_f1s, cc_f1s, model_labels = [], [], []

for mt, json_name in [('cnn', 'cnn_results.json'),
                       ('lstm', 'lstm_results.json'),
                       ('wav2vec', 'wav2vec_results.json')]:
    json_path = RESULTS_DIR / json_name
    if json_path.exists() and mt in all_loco:
        with open(json_path, encoding='utf-8') as f:
            ic = json.load(f)
        ic_f1s.append(ic['summary']['f1_macro_mean'])
        f1ms = [all_loco[mt][f'test_{c}']['f1_macro'] for c in DATASETS if f'test_{c}' in all_loco[mt]]
        cc_f1s.append(np.mean(f1ms))
        model_labels.append(mt.upper())

fig = go.Figure(data=[
    go.Bar(name='In-corpus', x=model_labels, y=ic_f1s, marker_color='steelblue'),
    go.Bar(name='Cross-corpus (LOCO)', x=model_labels, y=cc_f1s, marker_color='salmon'),
])
fig.update_layout(
    title='In-corpus vs Cross-corpus F1 (macro)',
    barmode='group',
    yaxis_title='F1 (macro)', yaxis_range=[0, 1],
    height=450, width=700, template='plotly_white',
)
fig.write_html(FIGURES_DIR / '11_in_vs_cross_corpus.html')
fig.write_image(FIGURES_DIR / '11_in_vs_cross_corpus.png', width=700, height=450, scale=2)
fig.show()